# Loan Data Processing

**Assignment:** B1 — Loan Data Processing

This notebook preprocesses the loan dataset for a classification task by:

- Loading and inspecting the data
- Cleaning currency formatting
- Fixing categorical inconsistencies
- Imputing missing values
- Removing duplicates
- Capping outliers using IQR
- Encoding labels
- Checking class balance
- Engineering new features
- Scaling numeric features using **RobustScaler**
- Saving the cleaned dataset

2) Import Libraries

In [17]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler

3) Load & Inspect

In [5]:
# Load dataset
df = pd.read_csv("raw_loan_dataset.csv")

print("First 5 Rows")
print(df.head())

print("\nDataset Info")
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

First 5 Rows
    Income  CreditScore  EmploymentYears LoanAmount HasCollateral  \
0   108810        537.0              1.1      25800           Yes   
1    96482        524.0             15.0      11200             Y   
2    28478          NaN              5.4      12100            No   
3  $25,851        561.0             17.6       7000           Yes   
4    38396        527.0              9.8      10700            No   

  PreviousDefaults  Approved  
0               No        No  
1               No       Yes  
2               No       Yes  
3               No       Yes  
4               No  Approved  

Dataset Info
<class 'pandas.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     str    
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     str  

4) Clean Currency Formatting

In [6]:
currency_columns = ["Income", "LoanAmount"]

for col in currency_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )

    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Currency cleaned successfully.\n")
print(df[currency_columns].head())

Currency cleaned successfully.

     Income  LoanAmount
0  108810.0     25800.0
1   96482.0     11200.0
2   28478.0     12100.0
3   25851.0      7000.0
4   38396.0     10700.0


5) Fix Category Errors

In [7]:
yes_no_map = {
    "yes": "Yes",
    "y": "Yes",
    "true": "Yes",
    "approved": "Yes",

    "no": "No",
    "n": "No",
    "false": "No",
    "rejected": "No"
}

columns = [
    "HasCollateral",
    "PreviousDefaults",
    "Approved"
]

for col in columns:

    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace(yes_no_map)
    )

    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))


HasCollateral
HasCollateral
No     51
Yes    49
NaN     2
yse     1
Name: count, dtype: int64

PreviousDefaults
PreviousDefaults
No     85
Yes    14
NaN     2
1       1
0       1
Name: count, dtype: int64

Approved
Approved
Yes    68
No     35
Name: count, dtype: int64


6) Impute Missing Values

In [8]:
# Numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Categorical columns
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

print("Missing Values After Imputation")
print(df.isnull().sum())

Missing Values After Imputation
Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64


/tmp/ipykernel_467630/2757696479.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df[col].fillna(df[col].median(), inplace=True)
/tmp/ipykernel_467630/2757696479.py:5: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works 

7) Remove Duplicates

In [9]:
before = len(df)

df = df.drop_duplicates()

after = len(df)

print("Rows Before:", before)
print("Rows After :", after)

Rows Before: 103
Rows After : 100


8) IQR Capping

In [10]:
outlier_columns = [
    "Income",
    "CreditScore",
    "LoanAmount",
    "EmploymentYears"
]

for col in outlier_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

print("Outliers capped successfully.")

Outliers capped successfully.


9) Label Encoding

In [11]:
label_map = {
    "Yes": 1,
    "No": 0
}

encode_cols = [
    "Approved",
    "HasCollateral",
    "PreviousDefaults"
]

for col in encode_cols:
    df[col] = df[col].map(label_map)

print("Approved Class Distribution")
print(df["Approved"].value_counts())

Approved Class Distribution
Approved
1    66
0    34
Name: count, dtype: int64


10) Class Balance

In [12]:
print("Class Counts")
print(df["Approved"].value_counts())

print("\nClass Proportions")
print(df["Approved"].value_counts(normalize=True))

Class Counts
Approved
1    66
0    34
Name: count, dtype: int64

Class Proportions
Approved
1    0.66
0    0.34
Name: proportion, dtype: float64


11) Feature Engineering

In [13]:
df["DebtToIncome"] = df["LoanAmount"] / df["Income"]

df["IncomePerYearEmployed"] = (
    df["Income"] /
    (df["EmploymentYears"] + 1)
)

print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  108810.0        537.0              1.1     25800.0            1.0   
1   96482.0        524.0             15.0     11200.0            1.0   
2   28478.0          NaN              5.4     12100.0            0.0   
3   25851.0        561.0             17.6      7000.0            1.0   
4   38396.0        527.0              9.8     10700.0            0.0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0               0.0         0      0.237111           51814.285714  
1               0.0         1      0.116084            6030.125000  
2               0.0         1      0.424889            4449.687500  
3               0.0         1      0.270783            1389.838710  
4               0.0         1      0.278675            3555.185185  


12) Scaling

In [18]:
numeric_features = [
    "Income",
    "CreditScore",
    "EmploymentYears",
    "LoanAmount",
    "DebtToIncome",
    "IncomePerYearEmployed"
]

scaler = RobustScaler()

df[numeric_features] = scaler.fit_transform(df[numeric_features])

print(df.head())

     Income  CreditScore  EmploymentYears  LoanAmount  HasCollateral  \
0  0.789411    -0.616766        -0.916335    0.240566            1.0   
1  0.543016    -0.694611         0.191235   -0.448113            1.0   
2 -0.816153          NaN        -0.573705   -0.405660            0.0   
3 -0.868658    -0.473054         0.398406   -0.646226            1.0   
4 -0.617926    -0.676647        -0.223108   -0.471698            0.0   

   PreviousDefaults  Approved  DebtToIncome  IncomePerYearEmployed  
0               0.0         0     -0.481283               7.467855  
1               0.0         1     -0.986630               0.152010  
2               0.0         1      0.302786              -0.100528  
3               0.0         1     -0.340686              -0.589461  
4               0.0         1     -0.307732              -0.243460  


13) Final Checks

In [15]:
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nFirst Five Rows")
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 95 non-null     float64
 1   CreditScore            95 non-null     float64
 2   EmploymentYears        95 non-null     float64
 3   LoanAmount             95 non-null     float64
 4   HasCollateral          97 non-null     float64
 5   PreviousDefaults       96 non-null     float64
 6   Approved               100 non-null    int64  
 7   DebtToIncome           90 non-null     float64
 8   IncomePerYearEmployed  90 non-null     float64
dtypes: float64(8), int64(1)
memory usage: 7.2 KB
None

Missing Values
Income                    5
CreditScore               5
EmploymentYears           5
LoanAmount                5
HasCollateral             3
PreviousDefaults          4
Approved                  0
DebtToIncome             10
IncomePerYearEmployed    10
dtype: int64


13) save dataset

In [20]:
# Save cleaned dataset
df.to_csv("clean_loan_dataset.csv", index=False)

print("✅ Clean dataset saved successfully!")

✅ Clean dataset saved successfully!
